# Notebook 1 — Noise models in medical imaging

The likelihood in Bayesian denoising/reconstruction **is** the noise model, so it
pays to *see* each one. Here we synthesise the four models from the theory
chapter on a phantom, and look at their histograms, signal-dependence, and SNR.

NumPy + Matplotlib only — nothing to install beyond the standard scientific stack.

You will build:
1. **Gaussian** — additive, signal-independent (the default).
2. **Poisson** — photon counting; variance = mean; SNR = √λ.
3. **Rician** — magnitude of a complex Gaussian (MRI); Rayleigh floor at zero signal.
4. **Speckle** — multiplicative (ultrasound); noise grows like √signal.

…plus two facts that matter later: **averaging improves SNR by √N**, and the
**Anscombe transform** turns Poisson into roughly constant-variance Gaussian.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(0)
plt.rcParams['figure.figsize'] = (10, 4)

def phantom(N=128):
    """A simple piecewise-constant phantom: background + disk + square.
    Intensities double as photon-count means for the Poisson demo."""
    yy, xx = np.mgrid[0:N, 0:N]
    img = np.full((N, N), 10.0)                    # background
    img[(xx - 45)**2 + (yy - 45)**2 < 28**2] = 60.0  # bright disk
    img[70:110, 70:110] = 35.0                     # mid-grey square
    return img

def snr(x):
    return x.mean() / x.std()

clean = phantom()
print('clean intensities:', np.unique(clean))
plt.imshow(clean, cmap='gray'); plt.title('clean phantom')
plt.colorbar(); plt.axis('off'); plt.show()

## 1. Gaussian noise (additive, signal-independent)

\(y_i = x_i + n_i,\quad n_i \sim \mathcal{N}(0,\sigma^2)\). The spread is the same
everywhere, regardless of how bright the pixel is — that is what "additive,
signal-independent" means, and it is why the negative log-likelihood is a plain
squared error.

In [ ]:
sigma = 8.0
gauss = clean + rng.normal(0.0, sigma, clean.shape)

fig, ax = plt.subplots(1, 3, figsize=(14, 4))
ax[0].imshow(clean, cmap='gray'); ax[0].set_title('clean'); ax[0].axis('off')
ax[1].imshow(gauss, cmap='gray'); ax[1].set_title(f'+ Gaussian (sigma={sigma})'); ax[1].axis('off')
ax[2].hist(gauss.ravel(), bins=80, color='steelblue')
ax[2].set_title('intensity histogram'); ax[2].set_xlabel('intensity')
plt.tight_layout(); plt.show()

# the noise (y - clean) has the SAME spread in dark and bright regions:
for level in [10.0, 35.0, 60.0]:
    resid = (gauss - clean)[clean == level]
    print(f'region intensity {level:5.1f}:  noise std = {resid.std():.2f}  (set sigma = {sigma})')

## 2. Poisson noise (photon counting, signal-dependent)

\(\Pr(X=k)=\lambda^k e^{-\lambda}/k!\), with \(\mathbb{E}[X]=\mathrm{Var}(X)=\lambda\)
and \(\mathrm{SNR}=\sqrt{\lambda}\). Treat each clean intensity as a mean photon
count λ and sample. Now the noise spread **grows with the signal** — bright
regions are noisier in absolute terms but cleaner in relative (SNR) terms.

In [ ]:
pois = rng.poisson(clean).astype(float)   # clean intensities act as lambda

fig, ax = plt.subplots(1, 3, figsize=(14, 4))
ax[0].imshow(pois, cmap='gray'); ax[0].set_title('Poisson counts'); ax[0].axis('off')

ax[1].scatter(clean.ravel()[::11], (pois - clean).ravel()[::11], s=3, alpha=0.25)
ax[1].set_xlabel('clean intensity (lambda)'); ax[1].set_ylabel('noise = y - lambda')
ax[1].set_title('noise spread grows with signal')

for L in [10, 35, 60]:
    s = rng.poisson(L, size=20000)
    ax[2].hist(s, bins=40, density=True, alpha=0.5, label=f'lam={L}, SNR~{np.sqrt(L):.1f}')
ax[2].legend(); ax[2].set_title('Poisson PMF; SNR = sqrt(lambda)')
plt.tight_layout(); plt.show()

# verify variance = mean per region, and that large lambda looks Gaussian:
for level in [10.0, 35.0, 60.0]:
    region = pois[clean == level]
    print(f'lambda {level:5.1f}:  sample mean {region.mean():6.2f}  sample var {region.var():6.2f}')

big = rng.poisson(200, size=50000)
xs = np.linspace(big.min(), big.max(), 200)
plt.hist(big, bins=60, density=True, alpha=0.5, label='Poisson(200)')
plt.plot(xs, np.exp(-(xs-200)**2/(2*200))/np.sqrt(2*np.pi*200), 'r-', label='N(200, 200)')
plt.legend(); plt.title('large lambda: Poisson approaches Gaussian'); plt.show()

## 3. Rician noise (MRI magnitude)

MRI measures a complex signal; each of the real and imaginary channels carries
i.i.d. Gaussian noise. The image we view is the **magnitude**
\(R=\sqrt{a^2+b^2}\), which is **Rician** distributed:
\[
p(x\mid\nu,\sigma)=\frac{x}{\sigma^2}\exp\!\Big(-\frac{x^2+\nu^2}{2\sigma^2}\Big)I_0\!\Big(\frac{\nu x}{\sigma^2}\Big),\quad x\ge 0.
\]
Because of the magnitude, the background (true signal ν ≈ 0) never reaches zero —
it follows a **Rayleigh** floor. We synthesise it directly from a complex
Gaussian and overlay the analytic PDF (using `np.i0`, the Bessel function I₀).

In [ ]:
sig = 8.0
real = clean + rng.normal(0.0, sig, clean.shape)   # true signal on the real channel
imag = 0.0   + rng.normal(0.0, sig, clean.shape)   # imaginary channel is pure noise
rician = np.sqrt(real**2 + imag**2)

fig, ax = plt.subplots(1, 3, figsize=(14, 4))
ax[0].imshow(rician, cmap='gray'); ax[0].set_title('Rician magnitude image'); ax[0].axis('off')

ax[1].hist(rician[clean == 10.0].ravel(), bins=60, density=True, color='salmon')
ax[1].set_title('low-signal voxels:\nRician shifted right of 0 (no true black)')
ax[1].set_xlabel('intensity')

xs = np.linspace(0, 95, 500)
for nu in [0, 10, 35, 60]:
    pdf = xs/sig**2 * np.exp(-(xs**2 + nu**2)/(2*sig**2)) * np.i0(nu*xs/sig**2)
    ax[2].plot(xs, pdf, label=f'nu={nu}' + (' (Rayleigh)' if nu == 0 else ''))
ax[2].legend(); ax[2].set_title('Rician PDF (nu=0 reduces to Rayleigh)')
plt.tight_layout(); plt.show()

print(f'background true signal = 10, but observed Rician mean = '
      f'{rician[clean==10.0].mean():.2f}  (note the upward bias)')

## 4. Speckle, averaging (√N), and the Anscombe transform

**Speckle** (a simple ultrasound-like model): \(Y = X + \sqrt{X}\,Z,\ Z\sim\mathcal{N}(0,\sigma^2)\) — multiplicative, so noise std grows like √signal.

**Averaging:** averaging \(N\) i.i.d. acquisitions multiplies SNR by √N — the workhorse of slow, clean MRI.

**Anscombe:** \(f(X)=2\sqrt{X+3/8}\) maps Poisson data to roughly **unit variance** (so you can pretend it is Gaussian, denoise, then invert).

In [ ]:
# --- speckle ---
sig_s = 1.5
speckle = np.clip(clean + np.sqrt(clean) * rng.normal(0.0, sig_s, clean.shape), 0, None)

fig, ax = plt.subplots(1, 3, figsize=(14, 4))
ax[0].imshow(speckle, cmap='gray'); ax[0].set_title('speckle (multiplicative)'); ax[0].axis('off')
ax[1].scatter(clean.ravel()[::11], (speckle - clean).ravel()[::11], s=3, alpha=0.25)
ax[1].set_xlabel('clean intensity'); ax[1].set_ylabel('noise'); ax[1].set_title('noise std ~ sqrt(signal)')

# --- averaging improves SNR by sqrt(N) ---
base_sigma = 12.0
Ns = [1, 4, 16, 64]
measured = []
for N in Ns:
    avg = np.mean([clean + rng.normal(0, base_sigma, clean.shape) for _ in range(N)], axis=0)
    measured.append((avg - clean).std())
ax[2].plot(Ns, measured, 'o-', label='measured noise std')
ax[2].plot(Ns, base_sigma/np.sqrt(Ns), 's--', label='predicted sigma/sqrt(N)')
ax[2].set_xlabel('N averaged'); ax[2].set_ylabel('residual std'); ax[2].legend()
ax[2].set_title('averaging: SNR improves by sqrt(N)')
plt.tight_layout(); plt.show()

# --- Anscombe variance stabilisation ---
lam_levels = np.array([1, 2, 5, 10, 20, 50, 100, 200])
raw_var, ans_var = [], []
for L in lam_levels:
    s = rng.poisson(L, size=40000).astype(float)
    raw_var.append(s.var())
    ans_var.append((2*np.sqrt(s + 3/8)).var())
plt.figure(figsize=(7, 4))
plt.plot(lam_levels, raw_var, 'o-', label='Var(Poisson) = lambda')
plt.plot(lam_levels, ans_var, 's-', label='Var(Anscombe) ~ 1')
plt.axhline(1, color='gray', ls=':'); plt.xlabel('lambda'); plt.ylabel('variance')
plt.legend(); plt.title('Anscombe stabilises Poisson variance'); plt.show()

## Exercises

1. For Poisson, plot measured SNR vs λ for several flat regions and confirm it tracks √λ.
2. Increase the Rician σ until the bright disk's histogram becomes visibly skewed; where does it stop looking Gaussian?
3. Denoise Poisson data two ways — (a) directly, (b) Anscombe → Gaussian denoise → inverse Anscombe — and compare (you'll need Notebook 2's denoiser).
4. Build a compound-Poisson sample (sum of a Poisson number of i.i.d. terms) and compare it to a matched Gaussian.

**Next:** Notebook 2 turns these likelihoods into a **MAP denoiser** with an MRF prior.